# Phase 4 — NLP Model & Detection Pipeline

**Input files (from previous notebooks):**
- `data/contracts_ie_clean.csv` — cleaned contracts + labels (NB01)
- `data/tfidf_matrix.npz` — sparse TF-IDF matrix (NB02a)
- `data/tfidf_vectorizer.pkl` — fitted vectorizer (NB02a)
- `data/embeddings.npy` — 768-dim sentence embeddings (NB02a)
- `data/contract_ids.csv` — row-order index for embeddings (NB02a)
- `data/ner_features.csv` — NER-based shared-address flag (NB02b)

**Outputs saved to `data/`:**
- `model_tfidf_lr.pkl` — TF-IDF + Logistic Regression baseline
- `model_tfidf_svc.pkl` — TF-IDF + LinearSVC baseline
- `model_emb_lr.pkl` — Embedding + LR classifier
- `anomaly_labels.csv` — IsolationForest anomaly predictions
- `metrics_summary.csv` — all model metrics in one table

---

## Strategy

We work across three layers of increasing complexity:

| Layer | Model | Features | Purpose |
|---|---|---|---|
| **Baseline** | LogisticRegression, LinearSVC | TF-IDF (sparse) | Fast interpretable benchmark |
| **Neural** | LogisticRegression | Sentence embeddings + structured | Semantic understanding |
| **Unsupervised** | IsolationForest, DBSCAN | Sentence embeddings | No labels needed |

The target label is `winner_concentration` — the only label with 100% coverage.
`single_bid` and `short_tender_period` are used where available (subset evaluation).

## 0. Colab / Local Setup

In [ ]:
import sys, subprocess, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    subprocess.run(['pip', 'install', '-q',
        'scikit-learn', 'umap-learn', 'joblib', 'tqdm', 'seaborn'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/fraud_detection_nlp/data/'
else:
    DATA_DIR = '../data'

os.makedirs(DATA_DIR, exist_ok=True)
print(f'DATA_DIR: {DATA_DIR}')

## 1. Imports

In [ ]:
import warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.sparse import load_npz, hstack

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, precision_recall_curve,
    average_precision_score, roc_auc_score, confusion_matrix
)
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print('Imports OK')

## 2. Load All Inputs

In [ ]:
# Main dataframe
df = pd.read_csv(os.path.join(DATA_DIR, 'contracts_ie_clean.csv'),
                 low_memory=False,
                 parse_dates=['publication_date', 'bid_deadline'])
print(f'Contracts loaded : {len(df):,}')

# TF-IDF matrix and vectorizer
X_tfidf = load_npz(os.path.join(DATA_DIR, 'tfidf_matrix.npz'))
tfidf   = joblib.load(os.path.join(DATA_DIR, 'tfidf_vectorizer.pkl'))
print(f'TF-IDF matrix    : {X_tfidf.shape}')

# Sentence embeddings — aligned to contracts via contract_ids.csv
emb_ids    = pd.read_csv(os.path.join(DATA_DIR, 'contract_ids.csv'))
embeddings = np.load(os.path.join(DATA_DIR, 'embeddings.npy'))
print(f'Embeddings       : {embeddings.shape}')

# NER features
ner = pd.read_csv(os.path.join(DATA_DIR, 'ner_features.csv'))
print(f'NER features     : {ner.shape}')

# Merge NER flag onto main dataframe
df = df.merge(ner[['contract_id', 'shared_address_flag']], on='contract_id', how='left')
df['shared_address_flag'] = df['shared_address_flag'].fillna(0).astype(int)

## 3. Feature Matrix Assembly

### Why add structured features on top of text?
Text alone misses numerical signals:
- A buyer who has won 2,000 contracts is suspicious regardless of what the description says.
- A contract with `copy_paste_description = 1` is suspicious regardless of keywords.

We stack three blocks horizontally:
1. **TF-IDF** — sparse keyword matrix (already computed)
2. **Sentence embeddings** — dense semantic matrix (already computed)
3. **Structured features** — normalised numerical signals

Models are trained on each combination so we can measure the contribution of each block.

In [ ]:
# Build structured feature block
# All columns must already exist in df after NB01 + NB02a
struct_cols = [
    'buyer_contracts_count',  # buyer activity volume
    'score_integrity',        # OpenTender composite integrity score
    'score_transparency',     # OpenTender composite transparency score
    'copy_paste_description', # similarity flag from NB02a
    'shared_address_flag',    # NER address flag from NB02b
]

X_struct_raw = df[struct_cols].fillna(0).values.astype(np.float32)

# Standardise: zero mean, unit variance
# Important for LR and SVC which are distance-based
scaler = StandardScaler()
X_struct = scaler.fit_transform(X_struct_raw)

print(f'Structured features shape: {X_struct.shape}')
print(f'Features: {struct_cols}')

## 4. Label Setup

We train on `winner_concentration` — 100% coverage, no NaN.

For `single_bid` and `short_tender_period` we evaluate on the subset of rows
where those labels are available, but do not train a separate model — the features
are the same and the model should generalise.

In [ ]:
y = df['winner_concentration'].values.astype(int)

print(f'Label: winner_concentration')
print(f'  Positive (flagged) : {y.sum():,} ({y.mean()*100:.1f}%)')
print(f'  Negative (clean)   : {(1-y).sum():,}')
print(f'  Total              : {len(y):,}')

In [ ]:
# Train/test split — stratified to preserve class balance
# Indices are shared across all feature representations
idx_train, idx_test, y_train, y_test = train_test_split(
    np.arange(len(y)), y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Train : {len(idx_train):,} rows  (pos={y_train.sum():,})')
print(f'Test  : {len(idx_test):,} rows  (pos={y_test.sum():,})')

## 5. Evaluation Helper

A single function runs every model through the same evaluation protocol
so results are directly comparable.

In [ ]:
all_metrics = []  # collect results for final summary table

def evaluate(name, model, X_test_block, y_test):
    """
    Evaluate a fitted model. Prints classification report and PR-AUC.
    Appends a row to all_metrics for the summary table.
    """
    y_pred = model.predict(X_test_block)

    # Probability or decision score for PR-AUC
    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(X_test_block)[:, 1]
    elif hasattr(model, 'decision_function'):
        y_score = model.decision_function(X_test_block)
    else:
        y_score = y_pred.astype(float)

    pr_auc = average_precision_score(y_test, y_score)
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

    p1  = report.get('1', {}).get('precision', 0)
    r1  = report.get('1', {}).get('recall', 0)
    f1  = report.get('1', {}).get('f1-score', 0)
    acc = report.get('accuracy', 0)

    print(f'\n=== {name} ===')
    print(classification_report(y_test, y_pred, target_names=['Clean', 'Flagged'], zero_division=0))
    print(f'PR-AUC: {pr_auc:.4f}')

    all_metrics.append({
        'model'    : name,
        'precision': round(p1, 4),
        'recall'   : round(r1, 4),
        'f1'       : round(f1, 4),
        'accuracy' : round(acc, 4),
        'pr_auc'   : round(pr_auc, 4),
    })

    return y_score

## 6. Baseline — TF-IDF Classifiers

### Why `class_weight='balanced'`?
The dataset has ~6% positive rate. Without correction, a model that predicts
"clean" for everything achieves 94% accuracy — but 0% recall on fraud.
`class_weight='balanced'` reweights the loss so each class contributes equally
to training, regardless of how many samples it has.

### Why both LR and LinearSVC?
LR outputs calibrated probabilities (needed for PR-AUC).
LinearSVC often achieves better precision on high-dimensional sparse data.
Comparing both tells us whether probability calibration costs accuracy.

In [ ]:
# TF-IDF feature slices for train/test
X_tfidf_train = X_tfidf[idx_train]
X_tfidf_test  = X_tfidf[idx_test]

# --- Logistic Regression on TF-IDF ---
lr_tfidf = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    C=1.0,
    solver='saga',   # efficient for large sparse matrices
    random_state=42
)
lr_tfidf.fit(X_tfidf_train, y_train)
scores_lr_tfidf = evaluate('LR + TF-IDF', lr_tfidf, X_tfidf_test, y_test)

joblib.dump(lr_tfidf, os.path.join(DATA_DIR, 'model_tfidf_lr.pkl'))
print('Saved: model_tfidf_lr.pkl')

In [ ]:
# --- LinearSVC on TF-IDF ---
svc_tfidf = LinearSVC(
    class_weight='balanced',
    max_iter=2000,
    C=1.0,
    random_state=42
)
svc_tfidf.fit(X_tfidf_train, y_train)
scores_svc_tfidf = evaluate('LinearSVC + TF-IDF', svc_tfidf, X_tfidf_test, y_test)

joblib.dump(svc_tfidf, os.path.join(DATA_DIR, 'model_tfidf_svc.pkl'))
print('Saved: model_tfidf_svc.pkl')

## 7. Embedding Classifier

We concatenate sentence embeddings (768-dim) with structured features (5-dim)
to give the model both semantic content and operational signals in one vector.

The embeddings are already L2-normalised. The structured features are StandardScaled.
Both are float32, so the concatenation is numerically stable.

In [ ]:
# Align embeddings to df row order using contract_ids
# emb_ids preserves the row order from NB02a
id_to_emb_idx = {cid: i for i, cid in enumerate(emb_ids['contract_id'])}
emb_row_idx   = df['contract_id'].map(id_to_emb_idx).values

# Fill any unmatched rows with zeros (should be 0 or very few)
valid_mask = ~pd.isna(emb_row_idx)
aligned_emb = np.zeros((len(df), embeddings.shape[1]), dtype=np.float32)
aligned_emb[valid_mask] = embeddings[emb_row_idx[valid_mask].astype(int)]

# Concatenate embeddings + structured features
X_neural = np.hstack([aligned_emb, X_struct]).astype(np.float32)
print(f'Neural feature matrix: {X_neural.shape}')

X_neural_train = X_neural[idx_train]
X_neural_test  = X_neural[idx_test]

In [ ]:
# --- LR on Embeddings + Structured ---
lr_emb = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    C=1.0,
    solver='lbfgs',   # works well for dense medium-dim input
    random_state=42
)
lr_emb.fit(X_neural_train, y_train)
scores_lr_emb = evaluate('LR + Embeddings + Structured', lr_emb, X_neural_test, y_test)

joblib.dump(lr_emb, os.path.join(DATA_DIR, 'model_emb_lr.pkl'))
print('Saved: model_emb_lr.pkl')

## 8. Precision-Recall Curves

PR curves are more informative than ROC for imbalanced datasets.
A high ROC-AUC can be achieved even with poor recall at low FPR;
PR-AUC directly measures the trade-off that matters for fraud detection:
how many true flags do we catch vs how many clean contracts do we incorrectly flag.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for name, scores in [
    ('LR + TF-IDF',                  scores_lr_tfidf),
    ('LinearSVC + TF-IDF',           scores_svc_tfidf),
    ('LR + Embeddings + Structured', scores_lr_emb),
]:
    p, r, _ = precision_recall_curve(y_test, scores)
    auc     = average_precision_score(y_test, scores)
    ax.plot(r, p, label=f'{name}  (AUC={auc:.3f})')

# Baseline: random classifier
ax.axhline(y=y_test.mean(), color='grey', linestyle='--', label='Random baseline')

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves — winner_concentration')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'plot_pr_curves.png'), dpi=150)
plt.show()

## 9. Anomaly Detection — Unsupervised

### Why unsupervised?
Our labels cover only a fraction of the fraud space. Anomaly detection operates
on all 139k contracts with no labels at all — it may surface fraud patterns
that the labelled indicators do not capture.

### IsolationForest
Isolates anomalies by randomly partitioning the feature space. Points that
are isolated in fewer splits are more anomalous. Fast and effective on
high-dimensional data. `contamination=0.05` tells the model to expect ~5%
anomalies, consistent with our labelled fraud rates.

### DBSCAN
Density-based clustering — contracts that do not belong to any dense cluster
are marked as outliers (`label = -1`). We run it on a UMAP-reduced 2D projection
to keep compute tractable and enable visual inspection.

In [ ]:
# IsolationForest on the full neural feature matrix
print('Fitting IsolationForest...')

iso = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)
iso.fit(X_neural)

# -1 = anomaly, 1 = normal  →  convert to 0/1
iso_pred   = iso.predict(X_neural)
iso_labels = (iso_pred == -1).astype(int)
iso_scores = -iso.score_samples(X_neural)  # higher = more anomalous

print(f'IsolationForest anomalies detected: {iso_labels.sum():,} ({iso_labels.mean()*100:.1f}%)')

# How well does it align with our winner_concentration label?
print('\nAlignment with winner_concentration label:')
print(classification_report(y, iso_labels,
      target_names=['Clean', 'Flagged'], zero_division=0))

In [ ]:
# UMAP dimensionality reduction for visualisation + DBSCAN
# We reduce from 773 dims to 2 dims for plotting
try:
    import umap
    print('Fitting UMAP (this takes a few minutes on CPU)...')
    reducer = umap.UMAP(n_components=2, random_state=42, n_jobs=-1)
    X_2d    = reducer.fit_transform(X_neural)
    print(f'UMAP done: {X_2d.shape}')
except ImportError:
    print('umap-learn not installed — using PCA fallback for 2D projection')
    from sklearn.decomposition import PCA
    X_2d = PCA(n_components=2, random_state=42).fit_transform(X_neural)
    print(f'PCA done: {X_2d.shape}')

In [ ]:
# DBSCAN on 2D projection
db = DBSCAN(eps=0.5, min_samples=10)
db_labels = db.fit_predict(X_2d)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_outliers = (db_labels == -1).sum()
print(f'DBSCAN clusters  : {n_clusters}')
print(f'DBSCAN outliers  : {n_outliers:,} ({n_outliers/len(db_labels)*100:.1f}%)')

In [ ]:
# Plot: 2D embedding coloured by winner_concentration label
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: ground truth labels
for label, colour, name in [(0, 'steelblue', 'Clean'), (1, 'tomato', 'Flagged')]:
    mask = y == label
    axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1],
                    c=colour, s=1, alpha=0.3, label=name)
axes[0].set_title('Ground truth: winner_concentration')
axes[0].legend(markerscale=5)

# Right: IsolationForest anomaly scores
sc = axes[1].scatter(X_2d[:, 0], X_2d[:, 1],
                     c=iso_scores, cmap='Reds', s=1, alpha=0.3)
plt.colorbar(sc, ax=axes[1], label='Anomaly score')
axes[1].set_title('IsolationForest anomaly scores')

plt.suptitle('2D projection (UMAP/PCA) of neural features', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'plot_embedding_clusters.png'), dpi=150)
plt.show()

In [ ]:
# Save anomaly predictions
anomaly_df = pd.DataFrame({
    'contract_id'         : df['contract_id'].values,
    'iso_forest_flag'     : iso_labels,
    'iso_forest_score'    : iso_scores.round(5),
    'dbscan_cluster'      : db_labels,
    'dbscan_outlier'      : (db_labels == -1).astype(int),
    'umap_x'              : X_2d[:, 0].round(5),
    'umap_y'              : X_2d[:, 1].round(5),
})

anomaly_df.to_csv(os.path.join(DATA_DIR, 'anomaly_labels.csv'), index=False)
print(f'Saved: anomaly_labels.csv  ({len(anomaly_df):,} rows)')

## 10. Error Analysis

We inspect false positives (clean contracts flagged as suspicious) and
false negatives (suspicious contracts missed) from the best supervised model.
This tells us where the model's understanding breaks down.

In [ ]:
# Use the LR + Embeddings model (typically best)
y_pred_best = lr_emb.predict(X_neural_test)

test_df = df.iloc[idx_test].copy()
test_df['y_true'] = y_test
test_df['y_pred'] = y_pred_best

false_positives = test_df[(test_df['y_true'] == 0) & (test_df['y_pred'] == 1)]
false_negatives = test_df[(test_df['y_true'] == 1) & (test_df['y_pred'] == 0)]

print(f'False positives : {len(false_positives):,}')
print(f'False negatives : {len(false_negatives):,}')

print('\n--- Sample FALSE POSITIVES (clean flagged as suspicious) ---')
print(false_positives[['title', 'buyer_name', 'buyer_contracts_count',
                        'winner_concentration']].head(5).to_string(index=False))

print('\n--- Sample FALSE NEGATIVES (suspicious missed) ---')
print(false_negatives[['title', 'buyer_name', 'buyer_contracts_count',
                        'winner_concentration']].head(5).to_string(index=False))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Clean', 'Flagged'],
            yticklabels=['Clean', 'Flagged'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix — LR + Embeddings + Structured')
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'plot_confusion_matrix.png'), dpi=150)
plt.show()

## 11. Metrics Summary

In [ ]:
metrics_df = pd.DataFrame(all_metrics)
metrics_df.to_csv(os.path.join(DATA_DIR, 'metrics_summary.csv'), index=False)

print('=== MODEL METRICS SUMMARY ===')
print(metrics_df.to_string(index=False))

## 12. Deliverables Summary

In [ ]:
deliverables = [
    ('model_tfidf_lr.pkl',       'Logistic Regression trained on TF-IDF'),
    ('model_tfidf_svc.pkl',      'LinearSVC trained on TF-IDF'),
    ('model_emb_lr.pkl',         'Logistic Regression trained on embeddings + structured'),
    ('anomaly_labels.csv',       'IsolationForest & DBSCAN predictions + UMAP coordinates'),
    ('metrics_summary.csv',      'All model metrics in one table'),
    ('plot_pr_curves.png',       'Precision-Recall curves'),
    ('plot_embedding_clusters.png', 'UMAP projection coloured by label / anomaly score'),
    ('plot_confusion_matrix.png','Confusion matrix for best supervised model'),
]

print(f'{"File":<40} {"Size":>10}  Description')
print('-' * 90)
for fname, desc in deliverables:
    fpath = os.path.join(DATA_DIR, fname)
    if os.path.exists(fpath):
        size  = os.path.getsize(fpath)
        sstr  = f'{size/1_048_576:.1f} MB' if size > 1_048_576 else f'{size/1024:.0f} KB'
        status = 'OK'
    else:
        sstr   = '--'
        status = 'MISSING'
    print(f'{fname:<40} {sstr:>10}  [{status}] {desc}')